# 🎬 VIDEO QUIZ GENERATOR STUDIO — GOOGLE COLAB RUNNER
Notebook này cho phép bạn khởi chạy toàn bộ Studio (Backend + Frontend Remotion) trực tiếp trên Google Colab có GPU/CPU miễn phí.
Chỉ gồm **3 bước (3 Cells)** đơn giản dưới đây:

In [ ]:
# ==============================================================================
# CELL 1: CLONE REPOSITORY TỪ GITHUB (HỖ TRỢ PRIVATE REPO QUA TOKEN)
# ==============================================================================
import os
import shutil

# --- [BƯỚC 1]: ĐIỀN THÔNG TIN REPOSITORY VÀ GITHUB TOKEN CỦA BẠN DƯỚI ĐÂY ---
# Ví dụ: GITHUB_REPO = "username/video-quiz-generator"
GITHUB_REPO = "YOUR_USERNAME/YOUR_REPOSITORY"  # <-- Điền username/repo tại đây

# Tạo token tại: GitHub -> Settings -> Developer settings -> Personal access tokens (classic)
# Quyền (scope) cần thiết: 'repo' (Full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN"  # <-- Điền Personal Access Token tại đây

# ------------------------------------------------------------------------------
REPO_NAME = GITHUB_REPO.split('/')[-1].replace('.git', '') if '/' in GITHUB_REPO else "Video-quiz-new"
WORKSPACE_DIR = f"/content/{REPO_NAME}"

print(f"[*] Đang chuẩn bị clone repository: {GITHUB_REPO}")

if os.path.exists(WORKSPACE_DIR) and os.path.exists(os.path.join(WORKSPACE_DIR, 'package.json')):
    print(f"[✓] Thư mục dự án đã tồn tại tại {WORKSPACE_DIR}. Tiến hành kéo cập nhật mới nhất (git pull)...")
    %cd {WORKSPACE_DIR}
    !git pull origin main || !git pull origin master
else:
    %cd /content
    if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN":
        AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    else:
        AUTH_URL = f"https://github.com/{GITHUB_REPO}.git"
    
    !git clone {AUTH_URL} {REPO_NAME}
    %cd {WORKSPACE_DIR}

print(f"\n[✓] Đã chuyển vào thư mục làm việc: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 2: KIỂM TRA & CÀI ĐẶT DEPENDENCY THÔNG MINH (SKIP NẾU ĐÃ CÓ / CÓ CACHE)
# ==============================================================================
import subprocess
import shutil
import os
import sys

def check_command(cmd_name):
    """Kiểm tra một lệnh CLI hệ thống đã tồn tại trong PATH hay chưa."""
    return shutil.which(cmd_name) is not None

def check_python_package(package_name):
    """Kiểm tra một Python package đã được cài đặt hay chưa."""
    try:
        __import__(package_name.replace('-', '_'))
        return True
    except ImportError:
        return False

print("=== KIỂM TRA MÔI TRƯỜNG & DEPENDENCIES ===")

# 1. Kiểm tra Node.js (Yêu cầu Node >= 18 cho Remotion & Vite)
node_ok = False
if check_command("node"):
    try:
        node_ver = subprocess.check_output(["node", "-v"]).decode().strip()
        major = int(node_ver.lstrip('v').split('.')[0])
        if major >= 18:
            print(f"[✓] Node.js đã sẵn sàng: {node_ver}")
            node_ok = True
    except Exception:
        pass

if not node_ok:
    print("[*] Đang cài đặt Node.js v20 LTS...")
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1
    print("[✓] Đã cài đặt Node.js thành công!")

# 2. Kiểm tra FFmpeg
if check_command("ffmpeg"):
    ffmpeg_ver = subprocess.check_output(["ffmpeg", "-version"]).decode().split('\n')[0]
    print(f"[✓] FFmpeg đã sẵn sàng: {ffmpeg_ver}")
else:
    print("[*] Đang cài đặt FFmpeg...")
    !apt-get update -qq && !apt-get install -y -qq ffmpeg > /dev/null 2>&1
    print("[✓] Đã cài đặt FFmpeg thành công.")

# 3. Kiểm tra Python packages (edge-tts)
python_pkgs = ["edge-tts"]
for pkg in python_pkgs:
    if check_python_package(pkg):
        print(f"[✓] Python package '{pkg}' đã sẵn sàng.")
    else:
        print(f"[*] Đang cài đặt Python package '{pkg}'...")
        !pip install -q {pkg}
        print(f"[✓] Đã cài đặt '{pkg}'.")

# 4. Kiểm tra & Cài đặt Node modules (Tận dụng Cache)
if os.path.exists("node_modules") and os.path.exists("node_modules/remotion"):
    print("[✓] Thư mục node_modules đã tồn tại. Bỏ qua npm install để tiết kiệm thời gian.")
else:
    print("[*] Đang cài đặt npm packages (sử dụng cache)... Chờ khoảng 1-2 phút...")
    !npm install --prefer-offline --no-audit --loglevel=error
    print("[✓] npm install hoàn tất!")


In [ ]:
# ==============================================================================
# CELL 3: KHỞI CHẠY SERVER NỀN & MỞ PUBLIC TUNNEL (DUY TRÌ LIÊN TỤC KHÔNG TẮT)
# ==============================================================================
import subprocess
import time
import urllib.request
import re
import os
import shutil
from IPython.display import display, HTML

# Cấu hình Port theo dự án
FRONTEND_PORT = 5400
BACKEND_PORT = 5410

print("=== KHỞI CHẠY BACKEND & FRONTEND ===")

# Dọn dẹp process cũ nếu có
!fuser -k 5400/tcp > /dev/null 2>&1 || true
!fuser -k 5410/tcp > /dev/null 2>&1 || true
!pkill -f cloudflared > /dev/null 2>&1 || true

# Khởi động Backend & Frontend ở chế độ background
log_file = open("/content/studio_server.log", "w")
server_proc = subprocess.Popen(
    ["npm", "run", "dev"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    shell=False
)

print("[*] Đang chờ Studio Video khởi động hoàn tất...")
server_ready = False
for attempt in range(40):
    time.sleep(2)
    try:
        with urllib.request.urlopen(f"http://localhost:{FRONTEND_PORT}", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        pass
    print(f"    ... đang kết nối (lần {attempt + 1}/40)")

if not server_ready:
    print("[!] CẢNH BÁO: Server chưa phản hồi kịp hoặc gặp lỗi. Xem log dưới đây:")
    !tail -n 25 /content/studio_server.log
else:
    print(f"[✓] Studio Video đã sẵn sàng trên cổng {FRONTEND_PORT}!")

# Cài đặt Cloudflare Tunnel (cloudflared) nếu chưa có
if not shutil.which("cloudflared"):
    print("[*] Đang tải Cloudflare Tunnel (cloudflared)... Miễn phí, không cần token!")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    print("[✓] Đã cài đặt cloudflared thành công.")

# Mở Public Tunnel cho Frontend (port 5400)
tunnel_log = "/content/cloudflared.log"
!rm -f {tunnel_log}
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{FRONTEND_PORT}"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT
)

print("[*] Đang tạo đường hầm Public URL an toàn...")
public_url = None
for _ in range(30):
    time.sleep(1.5)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

print("\n" + "=" * 65)
if public_url:
    print("🚀 ĐÃ MỞ THÀNH CÔNG STUDIO VIDEO QUIZ TRÊN GOOGLE COLAB!")
    print("=" * 65)
    print(f"\n👉 LINK TRUY CẬP WEB CỦA BẠN:\n   {public_url}\n")
    print("=" * 65)
    try:
        display(HTML(f'''
            <div style="background:#0f172a;border:2px solid #00e5ff;border-radius:12px;padding:20px;text-align:center;margin:15px 0;">
                <h3 style="color:#ffffff;margin:0 0 10px 0;">🎬 Studio Video Quiz Sẵn Sàng</h3>
                <a href="{public_url}" target="_blank" style="display:inline-block;background:#00e5ff;color:#000000;font-weight:bold;font-size:16px;padding:12px 28px;border-radius:8px;text-decoration:none;box-shadow:0 0 15px rgba(0,229,255,0.4);">
                    👉 BẤM VÀO ĐÂY ĐỂ MỞ STUDIO VIDEO
                </a>
                <p style="color:#94a3b8;font-size:13px;margin:10px 0 0 0;">(Link mở trên tab mới, hỗ trợ render trực tiếp)</p>
            </div>
        '''))
    except Exception:
        pass

    print("[*] 🔒 DUY TRÌ KẾT NỐI: Cell này sẽ GIỮ CHẠY LIÊN TỤC để bạn sử dụng Studio và Render video mà không bị ngắt quãng.")
    print("[*] ⚠️ VUI LÒNG KHÔNG BẤM NÚT DỪNG (STOP) CELL NÀY TRONG KHI ĐANG DÙNG HOẶC RENDER!")
    print("[*] Khi nào muốn dừng Studio, bạn mới bấm nút Dừng trên Colab.\n")

    start_time = time.time()
    last_ping = 0
    try:
        while True:
            time.sleep(3)
            elapsed = int(time.time() - start_time)
            if tunnel_proc.poll() is not None:
                print(f"[{time.strftime('%H:%M:%S')}] [!] Cloudflare tunnel bị ngắt. Đang tự động kết nối lại...")
                tunnel_proc = subprocess.Popen(
                    ["cloudflared", "tunnel", "--url", f"http://localhost:{FRONTEND_PORT}"],
                    stdout=open(tunnel_log, "a"),
                    stderr=subprocess.STDOUT
                )
            if server_proc.poll() is not None:
                print(f"[{time.strftime('%H:%M:%S')}] [!] Cảnh báo: Server dev bị dừng đột ngột. Chi tiết log cuối:")
                with open("/content/studio_server.log", "r") as f:
                    print(f.read()[-1000:])
                break
            if elapsed - last_ping >= 60:
                last_ping = elapsed
                mins = elapsed // 60
                print(f"[{time.strftime('%H:%M:%S')}] [Heartbeat] Studio & Tunnel đang hoạt động ổn định ({mins} phút) | Link: {public_url}")
    except KeyboardInterrupt:
        print("\n[✓] Đã nhận lệnh dừng từ người dùng. Tiến hành tắt Server và giải phóng Tunnel...")
        tunnel_proc.terminate()
        server_proc.terminate()
        print("[✓] Đã tắt an toàn!")
else:
    print("[!] Không tìm thấy Cloudflare URL. Kiểm tra log:")
    !cat {tunnel_log}
